# **Youtube RAG Chatbot**

In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-groq faiss-cpu tiktoken python-dotenv langchain-huggingface langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from langchain_groq import ChatGroq
from google.colab import userdata
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

## Step 1 : Indexing (Document Ingestion)

In [ ]:
!pip install regex

In [ ]:
# Test URL = https://www.youtube.com/watch?v=wjZofJX0v4M

In [ ]:
# Get the video_id from the video url
import re
video_url = input("Enter the Youtube URL:")
video_id_match = re.search(r"(?<=v=)[a-zA-Z0-9_-]+", video_url)

if video_id_match:
    video_id = video_id_match.group(0)
    print(video_id)
else:
    print("Could not find a video ID in the provided URL.")

Enter the Youtube URL:https://www.youtube.com/watch?v=wjZofJX0v4M
wjZofJX0v4M


In [ ]:
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

The initials GPT stand for Generative Pretrained Transformer. So that first word is straightforward enough, these are bots that generate new text. Pretrained refers to how the model went through a process of learning from a massive amount of data, and the prefix insinuates that there's more room to fine-tune it on specific tasks with additional training. But the last word, that's the real key piece. A transformer is a specific kind of neural network, a machine learning model, and it's the core invention underlying the current boom in AI. What I want to do with this video and the following chapters is go through a visually-driven explanation for what actually happens inside a transformer. We're going to follow the data that flows through it and go step by step. There are many different kinds of models that you can build using transformers. Some models take in audio and produce a transcript. This sentence comes from a model going the other way around, producing synthetic speech just from

## Step 1b: Text Splitting (Chunking)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

37

In [ ]:
chunks[0]

Document(metadata={}, page_content="The initials GPT stand for Generative Pretrained Transformer. So that first word is straightforward enough, these are bots that generate new text. Pretrained refers to how the model went through a process of learning from a massive amount of data, and the prefix insinuates that there's more room to fine-tune it on specific tasks with additional training. But the last word, that's the real key piece. A transformer is a specific kind of neural network, a machine learning model, and it's the core invention underlying the current boom in AI. What I want to do with this video and the following chapters is go through a visually-driven explanation for what actually happens inside a transformer. We're going to follow the data that flows through it and go step by step. There are many different kinds of models that you can build using transformers. Some models take in audio and produce a transcript. This sentence comes from a model going the other way around, 

## Step 1c and 1d: Embedding Creation of Chunks and Storing in Vectorstore

In [ ]:
embedding_model = HuggingFaceEndpointEmbeddings(
    repo_id = "ibm-granite/granite-embedding-97m-multilingual-r2",
    task = "feature-extraction",
    huggingfacehub_api_token = userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')
)

In [ ]:
vectorstore = FAISS.from_documents(
    documents = chunks,
    embedding = embedding_model
)

In [ ]:
vectorstore.index_to_docstore_id

{0: 'e9b14b6d-0df8-4d70-bce5-046e6b2b9074',
 1: '218cc22e-79f3-4cf4-be70-885600975e1d',
 2: '8542d9bc-3741-4e56-b6b3-935798400575',
 3: 'e6c6d47d-0c71-4bbe-be2a-27979715d674',
 4: '7b651cb3-d0c4-441d-b710-d412239aabad',
 5: '0a19f23d-7200-43dc-a77f-c227c9c2f10a',
 6: '0a6f9082-632d-4171-a85d-f3bc76d24ed2',
 7: '4adb3beb-c0b2-4975-a6e0-3ab2e3b70b75',
 8: '0c85eafa-a0d2-4ee6-8dee-2061281335c9',
 9: 'ecf43666-ab5e-44bd-86a8-2c1f06f28a8f',
 10: 'd7226ec5-f285-4972-bb5d-72d9298b9f56',
 11: '8016d133-2c5c-4fe6-a928-4d9d5e0f4eeb',
 12: 'dabaa003-7b30-438b-bfac-d50390c88b23',
 13: '5e8858ab-8ffe-4a4e-ba79-bcc780529b86',
 14: '71a6c5a5-cde1-4cab-8dcf-1210957c4c6e',
 15: 'eb3b412a-f7dc-4642-9a8f-1c321f83dcf3',
 16: '804b7192-a049-42ac-8022-04028209163b',
 17: '18ccc867-0d17-4777-8120-1be4357318f2',
 18: '36695a0c-56f9-4339-8470-20ebacc5493d',
 19: 'f989a9f4-548b-4b9d-bc5f-84c144d9031d',
 20: 'd70edad8-879e-4b0e-91ee-04c0265e5c3d',
 21: 'a3236b62-d8b1-4f6b-905b-e91ce38870ff',
 22: 'b7066d80-4b06-

In [ ]:
vectorstore.get_by_ids(['e9b14b6d-0df8-4d70-bce5-046e6b2b9074'])

[Document(id='e9b14b6d-0df8-4d70-bce5-046e6b2b9074', metadata={}, page_content="The initials GPT stand for Generative Pretrained Transformer. So that first word is straightforward enough, these are bots that generate new text. Pretrained refers to how the model went through a process of learning from a massive amount of data, and the prefix insinuates that there's more room to fine-tune it on specific tasks with additional training. But the last word, that's the real key piece. A transformer is a specific kind of neural network, a machine learning model, and it's the core invention underlying the current boom in AI. What I want to do with this video and the following chapters is go through a visually-driven explanation for what actually happens inside a transformer. We're going to follow the data that flows through it and go step by step. There are many different kinds of models that you can build using transformers. Some models take in audio and produce a transcript. This sentence com

## Step 2: Retrieval

In [ ]:
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 4}
)

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEndpointEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7e241818f0b0>, search_kwargs={'k': 4})

In [ ]:
relevant_chunks = retriever.invoke("What is attention?")

In [ ]:
for i,doc in enumerate(relevant_chunks):
  print(f"-----Chunk {i+1}-----")
  print("Content:",doc.page_content)

-----Chunk 1-----
Content: with the unembedding matrix, machine learning people would refer to the components in that raw, unnormalized output as the logits for the next word prediction. A lot of the goal with this chapter was to lay the foundations for understanding the attention mechanism, Karate Kid wax-on-wax-off style. You see, if you have a strong intuition for word embeddings, for softmax, for how dot products measure similarity, and also the underlying premise that most of the calculations have to look like matrix multiplication with matrices full of tunable parameters, then understanding the attention mechanism, this cornerstone piece in the whole modern boom in AI, should be relatively smooth. For that, come join me in the next chapter. As I'm publishing this, a draft of that next chapter is available for review by Patreon supporters. A final version should be up in public in a week or two, it usually depends on how much I end up changing based on that review. In the meantime

## Step 4: Augmentation

In [ ]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    api_key = userdata.get('GROQ_API_KEY')
)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question = "What do you mean by attention ?"
retrieved_docs = retriever.invoke(question)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'with the unembedding matrix, machine learning people would refer to the components in that raw, unnormalized output as the logits for the next word prediction. A lot of the goal with this chapter was to lay the foundations for understanding the attention mechanism, Karate Kid wax-on-wax-off style. You see, if you have a strong intuition for word embeddings, for softmax, for how dot products measure similarity, and also the underlying premise that most of the calculations have to look like matrix multiplication with matrices full of tunable parameters, then understanding the attention mechanism, this cornerstone piece in the whole modern boom in AI, should be relatively smooth. For that, come join me in the next chapter. As I\'m publishing this, a draft of that next chapter is available for review by Patreon supporters. A final version should be up in public in a week or two, it usually depends on how much I end up changing based on that review. In the meantime, if you want to dive int

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text='\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don\'t know.\n\n      with the unembedding matrix, machine learning people would refer to the components in that raw, unnormalized output as the logits for the next word prediction. A lot of the goal with this chapter was to lay the foundations for understanding the attention mechanism, Karate Kid wax-on-wax-off style. You see, if you have a strong intuition for word embeddings, for softmax, for how dot products measure similarity, and also the underlying premise that most of the calculations have to look like matrix multiplication with matrices full of tunable parameters, then understanding the attention mechanism, this cornerstone piece in the whole modern boom in AI, should be relatively smooth. For that, come join me in the next chapter. As I\'m publishing this, a draft of that next chapter is available for review 

## Step 5: Generation

In [ ]:
answer = llm.invoke(final_prompt)
answer.content

'In the text, **attention** refers to the part of a transformer that lets each token’s vector “talk to” the other token vectors.  \nThe attention block determines which surrounding words are relevant to a given word and how strongly they should influence that word’s representation. In other words, it computes weighted interactions (often via dot‑products and a softmax) so that the updated vector for each token reflects the context‑dependent meanings of the words around it.'

# **Building a Chain**

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def formatted_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel(
    {'context': retriever | RunnableLambda(formatted_docs),
     'question': RunnablePassthrough()}
)

In [ ]:
parallel_chain.invoke("What is attention?")

{'context': 'with the unembedding matrix, machine learning people would refer to the components in that raw, unnormalized output as the logits for the next word prediction. A lot of the goal with this chapter was to lay the foundations for understanding the attention mechanism, Karate Kid wax-on-wax-off style. You see, if you have a strong intuition for word embeddings, for softmax, for how dot products measure similarity, and also the underlying premise that most of the calculations have to look like matrix multiplication with matrices full of tunable parameters, then understanding the attention mechanism, this cornerstone piece in the whole modern boom in AI, should be relatively smooth. For that, come join me in the next chapter. As I\'m publishing this, a draft of that next chapter is available for review by Patreon supporters. A final version should be up in public in a week or two, it usually depends on how much I end up changing based on that review. In the meantime, if you want

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke("Can you summarize the video ?")

'**Video Summary**\n\nThe video is part of a short mini‑series that walks viewers through the fundamentals of deep learning with a focus on transformer models. It starts by grounding the discussion in the basic idea of machine learning—using data to learn a function that maps inputs (like images or text) to outputs (labels, next words, etc.)—and contrasts this data‑driven approach with explicit, rule‑based programming.\n\nThe speaker then explains how early large language models such as GPT‑3 worked: they generate text by sampling from a probability distribution, appending each new token, and repeating the process. The video describes how this sampling procedure is turned into a conversational chatbot by adding a “system prompt” that sets the context and then feeding the user’s initial query as the first dialogue turn.\n\nNext, the video shifts to the core of transformer architecture. It outlines the structure of a transformer network, highlighting the attention blocks that are conside

# **Tools in LangChain**

In [1]:
!pip install -U ddgs langchain-experimental langchain langchain-community langchain-core langchain-groq langchain-classic langchain-huggingface duckduckgo-search pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/

## 1. BuiltinTool - DuckDuckGo

In [7]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()
result = search_tool.invoke('waht is the latest version of python')
print(result)

April 15, 2026 - This version currently receives full bug-fix and security updates, while Python 3.13—released in October 2024—will continue to receive bug-fixes until October 2026, and after that will only receive security fixes until its end-of-life in 2029. 6 days ago - As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project's annual release cycle and five-year support policy. Python 3.15 is currently in the alpha development phase, and the stable release ... 4 days ago - The main branch is currently the future Python 3.16, and is the only branch that accepts new features. The latest release for each Python version can be found on the download page.(See below for a ... January 14, 2026 - The latest major version, Python 3.14 was officially released on October 7, 2025. 3 weeks ago - Every release gets: ... A Python release only supports a Windows platform while Microsoft considers the platform under extended support. Python 3

In [6]:
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults()
results = search.invoke("what is the latest version of Python")

print(results)

snippet: April 15, 2026 - Releases of Python 3 up through 3.12 include the 2to3 utility, which automates the translation of Python 2 code to Python 3. As of February 2026, Python 3.14.3 is the latest stable release., title: History of Python - Wikipedia, link: https://en.wikipedia.org/wiki/History_of_Python, snippet: 6 days ago - As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project's annual release cycle and five-year support policy. Python 3.15 is currently in the alpha development phase, and the stable release ..., title: Python (programming language) - Wikipedia, link: https://en.wikipedia.org/wiki/Python_(programming_language), snippet: 4 days ago - The main branch is currently the future Python 3.16, and is the only branch that accepts new features. The latest release for each Python version can be found on the download page.(See below for a ..., title: Status of Python versions, link: https://devguide.python.org/versio

## 2. BuiltinTool = Shell Tool

In [ ]:
from langchain_community.tools import ShellTool
shell_tool = ShellTool()

results = shell_tool.invoke("ls")
print(results)

Executing command:
 ls
sample_data



## Custom Tools

In [9]:
# Step1- Create a function
def multiply(a,b):
  """Multiply two numbers"""
  return a*b

In [10]:
# Step 2 add some  type hinting
def multiply(a: int, b: int) -> int:
  """Multiply two numbers"""
  return a*b

In [12]:
# Step 3 : Add @tool decorator
from langchain_community.tools import tool

@tool
def multiply(a: int, b: int) -> int:
  """Multiply two numbers when given a,b"""
  return a*b

In [13]:
result = multiply.invoke({"a":3, "b":2})
print(result)

6


In [14]:
multiply.name

'multiply'

In [15]:
multiply.description

'Multiply two numbers when given a,b'

In [16]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [18]:
# Now to bind this multiply tool with llm
from langchain_groq import ChatGroq
from google.colab import userdata

llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    api_key = userdata.get('GROQ_API_KEY')
)

In [19]:
llm_with_tools = llm.bind_tools([multiply])

In [20]:
llm_with_tools.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'We need to respond. The user says "Hi". Just greet.'}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 125, 'total_tokens': 158, 'completion_time': 0.034581716, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.006964581, 'prompt_tokens_details': None, 'queue_time': 0.031654482, 'total_time': 0.041546297}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_2caf65dc64', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7dca-aef9-7c21-a9c9-d20c7449d1f6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 125, 'output_tokens': 33, 'total_tokens': 158, 'output_token_details': {'reasoning': 15}})

In [21]:
llm_with_tools.invoke("Can you multiply 6 with 10")
# As we can see this and verify that yes LLM suggested that we can use this particular tool, and we must mention that LLM only suggests and never executes a tool and this is done by LangChain or developer

# tool_calls=[{'name': 'multiply', 'args': {'a': 6, 'b': 10}, 'id': 'fc_156c6a6d-8093-4a11-8207-9153ce1ca888', 'type': 'tool_call'}]

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to multiply 6 with 10. We have a function multiply. So call the function.', 'tool_calls': [{'id': 'fc_156c6a6d-8093-4a11-8207-9153ce1ca888', 'function': {'arguments': '{"a":6,"b":10}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 132, 'total_tokens': 181, 'completion_time': 0.050874757, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.006453265, 'prompt_tokens_details': None, 'queue_time': 0.031580476, 'total_time': 0.057328022}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a7e515d7c3', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e7dca-fb34-7cb1-a3c9-ef025af47b48-0', tool_calls=[{'name': 'multiply', 'args': {'a': 6, 'b': 10}, 'id': 'fc_156c6a6d-8093-4a11-8207-9153ce1ca888', 'type': 'tool_call'}], invalid_tool_calls=[], u

In [24]:
result = llm_with_tools.invoke("Can you multiply 6 with 10")

In [25]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 6, 'b': 10},
 'id': 'fc_c5dabf86-65cd-4983-a571-10ac7fd64e72',
 'type': 'tool_call'}

In [26]:
# Now If i will send this tool_calls into the tool again to execite it and  we will see a new message of ToolMessage() which LLM can use to produce answer
multiply.invoke(result.tool_calls[0])

ToolMessage(content='60', name='multiply', tool_call_id='fc_c5dabf86-65cd-4983-a571-10ac7fd64e72')

In [30]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# Add the user's query into HumanMessage
query = HumanMessage("What is the product of 3 and 100 ?")

#Create a list of chat history whatever is talked about
messages = [query]

# Now lets invoke the llm_with_tools with the first message
result = llm_with_tools.invoke(messages)
messages.append(result)
tool_called = result.tool_calls[0]
# Send the tool used args to Tool to get the result
tool_result = multiply.invoke(tool_called)
messages.append(tool_result)

#Finally send the whole history to the llm
answer = llm_with_tools.invoke(messages)
answer.content


'The product of 3 and 100 is **300**.'

# **Currency Converter**

In [309]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool, InjectedToolArg
import requests
from langchain_groq import ChatGroq
from typing import Annotated
from google.colab import userdata

In [310]:
# Lets first define the LLM tools

@tool
def get_conversion_rate(from_currency: str, to_currency: str) -> float:
  """Get the conversion rate from one currency to another"""
  result = requests.get(f"https://v6.exchangerate-api.com/v6/2c5a3d7a966def5a058cb83e/pair/{from_currency}/{to_currency}")
  return result.json()

@tool
def multiply(amount: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """This tool multiplies an amount from one currency to another by the help of conversion_rate"""
  return amount * conversion_rate


In [311]:
query = HumanMessage("What is the conversion rate from USD to INR, and also convert 87888.50 USD to INR")

In [312]:
history= [query]

In [313]:
history

[HumanMessage(content='What is the conversion rate from USD to INR, and also convert 87888.50 USD to INR', additional_kwargs={}, response_metadata={})]

In [314]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    api_key = userdata.get('GROQ_API_KEY')
)

In [315]:
llm_with_tools = llm.bind_tools([get_conversion_rate, multiply])

In [316]:
result1 = llm_with_tools.invoke(history)
history.append(result1)

In [317]:
result1

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use function get_conversion_rate to get conversion rate USD to INR. Then multiply to convert 87888.50 USD to INR. But the multiply function expects amount: number. But we need conversion rate. Possibly multiply uses the previously fetched rate? But the function signature: multiply({ amount: number }) => any. It might use internal state of last conversion rate? Not sure. We might need to call get_conversion_rate, then multiply with amount. But we need to pass amount only; maybe the function multiplies using last conversion rate. But the description: "This tool multiplies an amount from one currency to another by the help of conversion_rate". So we need to provide amount, but the function likely uses some internal conversion rate. However, we need to first call get_conversion_rate to get rate, then use multiply. The user wants both conversion rate and converted amount. So we should call get_conversion_rate with fro

In [318]:
result1.tool_calls

[{'name': 'get_conversion_rate',
  'args': {'from_currency': 'USD', 'to_currency': 'INR'},
  'id': 'fc_814975c9-b889-4f75-bfdf-2b3135cd9990',
  'type': 'tool_call'}]

In [319]:
import json

for tool_call in result1.tool_calls:
  if tool_call["name"] == "get_conversion_rate":
    tool_message1 = get_conversion_rate.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)["conversion_rate"]
    history.append(tool_message1)
  if tool_call["name"] == "multiply":
    tool_call['args']['conversion_rate'] = conversion_rate
    amount = multiply.invoke(tool_call)
    history.append(amount)

In [320]:
history

[HumanMessage(content='What is the conversion rate from USD to INR, and also convert 87888.50 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use function get_conversion_rate to get conversion rate USD to INR. Then multiply to convert 87888.50 USD to INR. But the multiply function expects amount: number. But we need conversion rate. Possibly multiply uses the previously fetched rate? But the function signature: multiply({ amount: number }) => any. It might use internal state of last conversion rate? Not sure. We might need to call get_conversion_rate, then multiply with amount. But we need to pass amount only; maybe the function multiplies using last conversion rate. But the description: "This tool multiplies an amount from one currency to another by the help of conversion_rate". So we need to provide amount, but the function likely uses some internal conversion rate. However, we need to first call get

In [321]:
result2 = llm_with_tools.invoke(history)

In [322]:
result2.tool_calls

[{'name': 'multiply',
  'args': {'amount': 87888.5},
  'id': 'fc_020bae24-5957-4f7e-aafb-83aed2ad7dc2',
  'type': 'tool_call'}]

In [323]:
import json

for tool_call in result2.tool_calls:
  if tool_call["name"] == "get_conversion_rate":
    tool_message1 = get_conversion_rate.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)["conversion_rate"]
    history.append(tool_message1)
  if tool_call["name"] == "multiply":
    tool_call['args']['conversion_rate'] = conversion_rate
    amount = multiply.invoke(tool_call)
    history.append(amount)

In [324]:
history

[HumanMessage(content='What is the conversion rate from USD to INR, and also convert 87888.50 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use function get_conversion_rate to get conversion rate USD to INR. Then multiply to convert 87888.50 USD to INR. But the multiply function expects amount: number. But we need conversion rate. Possibly multiply uses the previously fetched rate? But the function signature: multiply({ amount: number }) => any. It might use internal state of last conversion rate? Not sure. We might need to call get_conversion_rate, then multiply with amount. But we need to pass amount only; maybe the function multiplies using last conversion rate. But the description: "This tool multiplies an amount from one currency to another by the help of conversion_rate". So we need to provide amount, but the function likely uses some internal conversion rate. However, we need to first call get

In [325]:
llm_with_tools.invoke(history)

AIMessage(content='**Conversion rate (USD → INR)**  \n- 1\u202fUSD = **95.2043\u202fINR**\n\n**Converted amount**  \n- 87\u202f888.50\u202fUSD × 95.2043\u202fINR/USD ≈ **8\u202f376\u202f763.12\u202fINR**\n\n*(The rate is taken from the latest data returned by the currency‑conversion API.)*', additional_kwargs={'reasoning_content': 'We need to interpret the output. The tool returned the conversion rate 95.2043. Then the assistant attempted to call multiply but incorrectly used get_conversion_rate again? Actually second call: "functions.get_conversion_rate to=assistant 8367363.12055" That\'s wrong; maybe they attempted to use multiply but wrote wrong call. We need to provide conversion rate and converted amount. 87888.50 USD * 95.2043 = 87888.5 * 95.2043. Let\'s compute: 87888.5 * 95.2043. 87888.5 * 95 = 835, (actually 87,888.5 * 95 = 8,358,807.5). Then add 0.2043 * 87,888.5 = 17,938. (Wait compute precisely). Let\'s compute using Python mental: 87,888.5 * 95.2043 = 87,888.5 * (95 + 0.20